# N2 · 自检一份开题报告草稿 (Audit Your Proposal Draft)

> 配套 L3-L4 · **真实科研动作**: 写一版开题报告七块骨架(哪怕是雏形), 用 `proposal_audit.py`
> 自动检查漏项/薄弱项, 再把检查结果转成开题答辩最可能被追问的问题清单。

In [1]:
import sys
from pathlib import Path
SRC = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC))
import proposal_audit as pa
print("七块骨架:", [name for _, name, _ in pa.SECTIONS])

七块骨架: ['研究背景与动机', '文献综述与研究缺口', '具体研究问题/假设', '初步方法设想', '时间线与里程碑', '风险与应对预案', '预期贡献']


## 1. 写一版完整的proposal(七块都填)

In [2]:
good = pa.blank_proposal()
good["background"] = "长上下文推理在超过32k token后普遍出现注意力稀释, 但该现象缺乏系统量化。"
good["gap"] = "现有工作只报告了端到端准确率下降, 未定位是注意力层还是位置编码的问题。"
good["question"] = "假设: 在64k token设定下, 注意力稀释导致的准确率下降比位置编码外推误差高≥2倍。"
good["method"] = "构造分层探针实验, 分别固定注意力/位置编码为理想值, 对比准确率恢复幅度。"
good["timeline"] = "第1-2月复现baseline; 第3-4月完成探针实验; 第5月出第一版结果; 第6月开始写作。"
good["risks"] = "若探针实验无法分离两个因素, 备选方案是改用因果干预(activation patching)定位。"
good["contribution"] = "首次量化区分长上下文两种退化机制的相对贡献, 指导后续工程优化优先级。"

print(pa.render(good))

=== 开题报告骨架 ===

## 研究背景与动机
长上下文推理在超过32k token后普遍出现注意力稀释, 但该现象缺乏系统量化。

## 文献综述与研究缺口
现有工作只报告了端到端准确率下降, 未定位是注意力层还是位置编码的问题。

## 具体研究问题/假设
假设: 在64k token设定下, 注意力稀释导致的准确率下降比位置编码外推误差高≥2倍。

## 初步方法设想
构造分层探针实验, 分别固定注意力/位置编码为理想值, 对比准确率恢复幅度。

## 时间线与里程碑
第1-2月复现baseline; 第3-4月完成探针实验; 第5月出第一版结果; 第6月开始写作。

## 风险与应对预案
若探针实验无法分离两个因素, 备选方案是改用因果干预(activation patching)定位。

## 预期贡献
首次量化区分长上下文两种退化机制的相对贡献, 指导后续工程优化优先级。


## 2. 完整性自检

In [3]:
chk = pa.audit(good)
print("✅ 骨架完整" if chk["ready"] else chk["issues"])

✅ 骨架完整


## 3. 反面: 敷衍的proposal(常见新手版本)

In [4]:
bad = pa.blank_proposal()
bad["background"] = "长上下文很重要。"
bad["question"] = "研究长上下文推理问题。"
bad["timeline"] = "预计一年完成。"
for i in pa.audit(bad)["issues"]:
    print("⚠", i)
print("\n→ 缺文献缺口定位 + 研究问题不可证伪 + 时间线模糊 + 无风险预案,"
      " 这是评审小组见得最多、也最容易被戳穿的四类问题。")

⚠ 缺「文献综述与研究缺口」这一节 —— 指出具体哪篇/哪类工作没做到什么, 而不是泛泛的文献罗列
⚠ 「具体研究问题/假设」看起来不够可证伪 —— 没有出现比较级/阈值类表述, 检查是否只是一句模糊的研究方向陈述而非具体假设
⚠ 缺「初步方法设想」这一节 —— 不要求完整方案, 但要有具体到能被追问细节的雏形
⚠ 「时间线与里程碑」用词过于模糊(如'尽快'/'预计一年') —— 改成带具体阶段和核对点的里程碑
⚠ 缺「风险与应对预案」这一节 —— 至少列出方法可能失败的场景, 以及每个风险对应的B计划
⚠ 缺「预期贡献」这一节 —— 假装做完了, 用一句话说出结论会让人'哦!'还是'so what?'

→ 缺文献缺口定位 + 研究问题不可证伪 + 时间线模糊 + 无风险预案, 这是评审小组见得最多、也最容易被戳穿的四类问题。


## 4. L4用: 把薄弱环节转成开题答辩追问预判

In [5]:
print("如果拿这份敷衍proposal去答辩, 评审最可能追问:")
for q in pa.defense_focus(bad):
    print("  ?", q)
print("\n→ 这正是L4「开题答辩」的准备核心: 与其等被问, 不如自己先用同一套检查跑一遍,"
      " 主动亮出风险预案, 比回避风险更能建立评审信任。")

如果拿这份敷衍proposal去答辩, 评审最可能追问:
  ? 这个缺口真的没人做过吗? 你查得够全面吗?
  ? 如果实验结果和你预期相反, 你打算怎么解读?
  ? 如果这个方法不work, 你的备选方案是什么?
  ? 如果第一年拿不到预期结果, 后面时间还够吗?
  ? 这个方向最可能失败在哪一步? 你想过吗?
  ? 假设做完了, 谁会真的在意这个结论?

→ 这正是L4「开题答辩」的准备核心: 与其等被问, 不如自己先用同一套检查跑一遍, 主动亮出风险预案, 比回避风险更能建立评审信任。


## 5. 反思(本专题L1-L4收官)

你刚完整走了一遍: 打分选方向(L1/N1) → 项目可行性(L2) → 写proposal并自检(L3/N2) → 预判答辩追问(L4)。带走:

- proposal不是idea卡的放大版, 是给评审看的正式文档, 七块骨架缺一不可。
- 可证伪的研究问题 + 具体里程碑 + 诚实的风险预案, 是最容易被忽视但最影响评审信任的三块。
- 开题答辩被打回不是失败(呼应 L0「回路A」), 是流程设计的校验点在正常工作——退回L1重新收窄,
  比跑完两年实验才发现方向有问题, 代价小得多。

> **本专题(research-direction-proposal)收官**: L0地图 → L1选方向 → L2评可行性 → L3写proposal → L4扛答辩。
> 加上已有9个科研技能专题的16个环节, 20环节全流程地图至此完整落地。